[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diogoflim/AM/blob/main/05_Reg_linear.ipynb)


# Introdução ao Aprendizado de máquina

**Professor: Diogo Ferreira de Lima Silva**

**TPP - UFF**

## Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

%matplotlib inline
np.random.seed(42)

# Regressão

Em tarefas de regressão, deseja-se aprender uma função que gera um rótulo com valor numérico (ex: um número real) dado alguma entrada.

Para resolver essa tarefa, o algoritmo de aprendizagem precisa gerar uma função:

$$\hat{y} = f:\mathbb{R}^n→\mathbb{R}$$

## Regressão Linear

Em um modelo de regressão linear, a função aprendida $f$ apresenta linearidade em termos dos parâmetros.

Exemplo:

$$f_{\vec{w},b}(\vec{x})=w_1x_1+w_2x_2+...+w_nx_n + b$$

$\vec{w}$ e $b$ são os parâmetros (coeficientes, pesos) que desejamos aprender


## Regressão Linear Simples

Para entender a intuição de modelos de regressão linear, iniciaremos com o caso mais simples, quando há apenas um atributo x.

**Esse modelo é chamado de regressão linear simples.**

Exemplos:

- entender o relacionamento entre os preços de títulos financeiros e o valor do dólar;  
- prever o consumo de energia elétrica com base no tamanho da fábrica.

# Gerando um conjunto de dados sintético

Vamos supor que desejamos prever o **preço de apartamentos** a partir da **área construída**.

Para fins didáticos, vamos gerar dados artificiais.  

A ideia será:

- gerar áreas de apartamentos;
- associar um preço aproximadamente linear;
- adicionar ruído aleatório para simular a variabilidade do mundo real.

In [ ]:
# área em m²
n_apartamentos = 120

area = np.random.normal(loc=150, scale=35, size= (n_apartamentos,1))
area = np.clip(area, 45, None)  # evitando áreas negativas ou muito pequenas

# modelo "verdadeiro" subjacente (desconhecido em problemas reais)
preco_base = 80 + 4.5 * area

# ruído aleatório
ruido = np.random.normal(loc=0, scale=35, size=(n_apartamentos,1))

# preço observado
preco = preco_base + ruido

dados = pd.DataFrame(np.concatenate((area,preco), axis=1), columns = ["area_m2", "preco_mil"])

dados.head()

### Visualização dos dados

Para visualizar nossos dados, criaremos um gráfico de dispersão

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(dados["area_m2"], dados["preco_mil"], alpha=0.75)
plt.xlabel("Área (m²)")
plt.ylabel("Preço (mil unidades monetárias)")
plt.title("Relação entre área e preço")
plt.show()

### Interpretação Estatística

**Ideia central**: É impossível obter uma reta que passe por todos os pontos, porém podemos perceber um relacionamento linear.

Ou seja, existe uma indicação de que os pontos estariam dispostos aleatoriamente em volta de uma reta.

Assim, seria razoável assumir que a média da variável aleatória $y$ está relacionada à variável explanatória $x$ por um relacionamento linear

$$E(y│x)=μ_{y|x}=w_0+w_1 x$$

- $w_0$ é o intercepto da equação
- $w_1$ é o coeficiente angular.

**Interpretação**: Embora a média de y seja uma função linear de $x$, um valor observado qualquer $y^{(i)}$ não cai necessariamente precisamente na reta.

$$y^{(i)}=w_0+w_1x^{(i)} + \epsilon$$

Assim, precisamos estimar $w_0$ e $w_1$ de modo a obter o modelo menos custoso.


### Visualizando diferentes retas

Vejamos como diferentes retas se ajustariam ao nosso conjunto de dados

In [ ]:
X_plot = np.linspace(dados["area_m2"].min(), dados["area_m2"].max(), 200)

plt.figure(figsize=(9,5))
plt.scatter(dados["area_m2"], dados["preco_mil"], alpha=0.55, label="dados")
plt.plot(X_plot, 2.5 * X_plot + 120, label="reta 1: y = 2.5x + 120")
plt.plot(X_plot, 4.5 * X_plot + 80, label="reta 2: y = 4.5x + 80")
plt.plot(X_plot, 6.0 * X_plot - 80, label="reta 3: y = 6.0x - 80")
plt.xlabel("Área (m²)")
plt.ylabel("Preço (mil)")
plt.title("Nem toda reta ajusta bem os dados")
plt.legend()
plt.show()

### Erro Médio Quadrático

Fazemos isso minimizando a função custo associada ao erro médio quadrático.

$$min_{w,b}⁡ \ J= \frac{1}{m} \sum_{i=1}^m [( wx^{(i)}+b) - y^{(i)}]^2$$

In [ ]:
# Impementaremos abaixo a função custo, calculada a partir de um loop sobre m exemplos

def custo_quadratico_medio (x, y, w, b):
    m = x.shape[0]
    custo = 0
    for i in range(m):
        f_wb = w * x[i] + b
        custo = custo + (f_wb - y[i])**2
    J = custo / m
    return J

In [ ]:
# testando três retas

retas = [
    ("reta 1", 2.5, 120),
    ("reta 2", 4.5, 80),
    ("reta 3", 6.0, -80)]

for nome, w, b in retas:
    #y_pred = prever(dados["area_m2"].values, w, b)
    print(f"{nome}: MSE = {custo_quadratico_medio(area, preco, w, b)}")

### Como encontrar as melhores estimativas de $w$ e $b$?

Sabemos que a nossa função custo é convexa em relação aos parâmetros!

**Estratégias Possíveis**:

1. Buscar a solução fechada (exata)

- A função pode ser minimizada com o sistema de derivadas parciais igualadas a zero

2. Gradiente descendente

### Solução Fechada

Podemos minimizar $J$, pelas condições de primeira ordem de otimização.

Escolheremos as estimativas de parâmetros que satisfaçam:

$$\frac{\partial J}{\partial w_0} = 0$$

$$\frac{\partial J}{\partial w_1} = 0$$


Como visto na sala de aula, a solução do sistema acima resulta nas fórmulas a seguir (presentes em livros de estatística):


$${\hat{w}} = \frac{\sum_{i=1}^{m}{y^{(i)}x^{(i)}}-\frac{\left(\sum_{i=1}^{m}y^{(i)}\right)\left(\sum_{i=1}^{m}x^{(i)}\right)}{m}}{\sum_{i=1}^{m} {x^{(i)}}^{2} -\frac{({\sum_{i=1}^{m}{x^{(i)})}}^2}{m}}$$


$${\hat{b}}=\bar{y} - {\hat{w}} \bar{x}$$


Assim, obtém-se o modelo regressor $$\hat{y}={\hat{w}}x + \hat{b}$$ estima o valor médio do modelo de regressão.

Cada observação satisfaz a relação $y^{(i)}={\hat{b}}+{\hat{w}}x+\epsilon$, onde $\epsilon=y^{(i)}-{\hat{y}}^{(i)}$ é chamado de desvio ou resíduo.

In [ ]:
X = area
y = preco

In [ ]:
m= X.shape[0] # número de exemplos

# Perceba que apesar da expressão do w parecer complexa, pode-se implementá-la em uma linha de código
w_hat = (y.T @ X - (np.sum(y) * np.sum(X)) / m) / (X.T @ X - np.sum(X)**2/m)

b_hat = np.mean(y) - w_hat * np.mean(X)

print(w_hat)
print(b_hat)

In [ ]:
prever = lambda x, w, b: w * x + b  # criando uma função para prever o y

y_ajustado = prever(X, w_hat, b_hat)

# Visualização
plt.figure(figsize=(8,5))
plt.scatter(X, y, alpha=0.7, label="dados")
plt.plot(X, y_ajustado, linewidth=2.5, label="reta ajustada")
plt.xlabel("Área (m²)")
plt.ylabel("Preço (mil)")
plt.title("Regressão linear simples ajustada aos dados")
plt.legend()
plt.show()

In [ ]:
#Calculando o custo nas estimativas!

custo_quadratico_medio(X, y, w_hat, b_hat).round(2)

### Usando a biblioteca sklearn

Assim como visto para outros métodos de classificação, a biblioteca *sci-kit learn* inclui o procedimento de **regressão linear**.


In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression() # instancia o modelo
lin_reg.fit(X, y) # ajuste do modelo aos dados
lin_reg.coef_, lin_reg.intercept_

Perceba que obtivemos os mesmos valores!

## Gerando exemplos de teste

Até o momento, utilizamos um conjunto de dados (X, y) para aprender os parâmetros de uma regressão linear. Podemos entender X como a matriz atributos de um conjunto de treinamento e y como um vetor de rótulos para os exemplos de treinamento.

Vamos usar uma metodologia similar para gerar valores para teste!

In [ ]:
# área em m²
n_teste = 30
X_teste = np.random.normal(loc=150, scale=35, size=(n_teste,1))
X_teste = np.clip(X_teste, 45, None)

# modelo "verdadeiro" subjacente (desconhecido em problemas reais)
y_teste = 80 + 4.5 * X_teste + np.random.normal(loc=0, scale=35, size=(n_teste,1))

Vamos visualisar o conjunto de teste

In [ ]:
pd.DataFrame(np.concatenate((X_teste,y_teste), axis=1), columns = ["Área", "Preço (1000's)"]).round(2)

### Usando a reta aprendida para prever rótulos

Usando os parâmetros estimados acima em lin_reg, podemos prever os preços do conjunto de teste.

In [ ]:
lin_reg.predict(X_teste)

**O custo pode ser calculado com nossa função custo total**

In [ ]:
# Usando os parâmetros do sklearn
print (custo_quadratico_medio (X_teste, y_teste, lin_reg.coef_, lin_reg.intercept_))

# Usando os parâmetros da nossa fórmula da função fechada
print (custo_quadratico_medio (X_teste, y_teste, w_hat, b_hat))

Também podemos calcular o custo com o sklearn.

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
mean_squared_error(lin_reg.predict(X_teste), y_teste)

------------------------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------------------------

## Regressão linear com vários atributos

No caso genérico, uma observação, então, é dada por um vetor de $d$ atributos $\mathbf{x}_i \in \mathbb{R}^d$.

Considerando o relacionamento entre as variáveis de entrada $\mathbf{x}_i \in \mathbb{R}^d$ e o valor de saída $y\in\mathbb{R}$ linear, temos uma **regressão linear**:

$$y=\mathbf{w}^{T}\mathbf{x} + \epsilon$$


Onde $\mathbf{w}^{T}=[w_1,\ldots,\ w_n]$ é um vetor de parâmetros, no qual $w_j$ é o coeficiente que multiplica o atributo $x_j$ antes de somar as contribuições de todos os atributos.

O coeficiente $w_j$ indica como a variável dependente $y$ muda em média quando $x_j$ é adicionado em uma unidade e as demais variáveis independentes permanecem constantes.

Esses parâmetros são valores que controlam o comportamento do sistema, muitas vezes chamados de pesos ou coeficientes da regressão.

### Adicionando o intercepto

Frequentemente, o termo regressão linear é usado para um modelo um pouco mais sofisticado, com um parâmetro adicional: o intercepto $b$.


Teríamos: $y=\mathbf{w}^T\mathbf{x}+b$.

No entanto, podemos continuar usando o modelo anterior (apenas com pesos). Para isso, inserimos um valor extra para cada observação $\vec{x}^{(i)}$, sempre igual a $x_{0}=1$.

Dessa forma, o peso correspondente à entrada extra ($w_0$) desempenha o papel do intercepto.


### Interpretação

O modelo com $n$ atributos descreve um hiperplano no espaço n-dimensional das variáveis dos coeficientes.

### Solução Exata


$$\nabla_\mathbf{w}\left|\left|\mathbf{y}-\hat{\mathbf{y}}\right|\right|_2^2 = 0\rightarrow\nabla_\mathbf{w}\left|\left|\mathbf{y}-\mathbf{X}\hat{\mathbf{w}}\right|\right|_2^2=0$$


$$
\nabla_\mathbf{w}\left(\mathbf{y}-\mathbf{X}\hat{\mathbf{w}}\right)^T\left(\mathbf{y}-\mathbf{X}\hat{\mathbf{w}}\right)=0
$$

$$
\nabla_\mathbf{w}\left(\mathbf{y}^\mathbf{T}\mathbf{y}-2{\hat{\mathbf{w}}}^T\mathbf{X}^\mathbf{T}\mathbf{y}+{\hat{\mathbf{w}}}^T\mathbf{X}^\mathbf{T}\mathbf{X}\hat{\mathbf{w}}\right)=0
$$

$$
-2\mathbf{X}^\mathbf{T}\mathbf{y}+2\mathbf{X}^\mathbf{T}\mathbf{X}\hat{\mathbf{w}}=0
$$


$$
2\mathbf{X}^\mathbf{T}\mathbf{X}\hat{\mathbf{w}}=2\mathbf{X}^\mathbf{T}\mathbf{y}
$$


$$
\mathbf{X}^\mathbf{T}\mathbf{X}\hat{\mathbf{w}}=\mathbf{X}^\mathbf{T}\mathbf{y}
$$

$$
\hat{\mathbf{w}}=\left(\mathbf{X}^\mathbf{T}\mathbf{X}\right)^{-1}\mathbf{X}^\mathbf{T}\mathbf{y}
$$


**Esse resultado é conhecido como Equação Normal**

Relembrando nossa matriz de atributos que contém apenas uma coluna (área construída).

In [ ]:
X[:10]

Vamos adicionar o valor 1 na primeira coluna da matriz X

In [ ]:
X_novo = np.column_stack([np.ones(len(X)), X])
X_novo[:10]


**A equação normal pode ser aplicada com uma única linha de código!!**

Perceba que chegaremos aos mesmos parâmetros aprendidos com a regressão linear simples acima

In [ ]:
#Equação normal em uma linha de código:

w_hat2 = np.linalg.inv(X_novo.T.dot(X_novo)).dot(X_novo.T).dot(y)

print(w_hat2)

In [ ]:
print (f"Regressão Linear Simples:  valor de b é {b_hat},  valor de w é {w_hat}")

print (f"Equação Normal: valor de w_0:{w_hat2[0]}, valor de w: {w_hat2[1]}")